<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="/ipynb/zh-CN/Computer-Science/Computer-Networks/07-naming-applications-and-content-delivery.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Computer Networks guideline](Computer-Networks.html)


## **Naming, Applications, and Content Delivery**

The previous chapters built the path beneath an application: IP forwarding reaches a host, routing selects paths, TCP or QUIC supplies transport behavior, and congestion control decides how aggressively shared capacity may be used. Applications still need answers to a different set of questions. What name identifies the service? What bytes constitute one request? Which operations are safe to retry? Can a cached representation replace an origin request? Should a video player wait for high quality or switch to a smaller segment before its playback buffer empties?

This chapter follows one web request from a typed URL to a delivered representation. The browser parses a URL, asks DNS for service location, chooses an application transport, sends an HTTP request, and may receive the response from a nearby cache rather than the origin. The same design vocabulary then explains store-and-forward email, message queues, adaptive streaming, real-time media, and peer-to-peer overlays.

::: {.callout-note}
On a first reading, separate **identity**, **protocol semantics**, and **placement**. DNS maps a name to information used to reach a service. HTTP defines what a request and response mean. Caches, proxies, and CDNs decide where a valid representation can be served. These layers cooperate, but none is a synonym for another.
:::

The cryptographic mechanisms behind HTTPS, DNSSEC, certificate validation, secure cookies, and WebRTC media protection belong to Chapter 8. This chapter identifies where those protections attach while concentrating on application behavior and content delivery.

### **Application-Layer Architecture**

#### **Client-Server Systems**

In a **client-server** architecture, an always-reachable service endpoint accepts requests from clients. The service may appear as one hostname while actually running behind load balancers, replicas, databases, and several regions. "Server" therefore describes a role in an exchange, not necessarily one physical machine.

Client-server systems centralize policy, durable state, access control, and observability. A bank can validate every transfer against one authoritative ledger; a web origin can produce a canonical representation. The trade-off is concentrated capacity and failure responsibility. Replication and a CDN improve scale, but the provider must provision and coordinate them.

#### **Peer-to-Peer Systems**

In a **peer-to-peer (P2P)** architecture, participating endpoints can both request and contribute resources. New peers can add upload capacity as they add demand, which is attractive for distributing large immutable objects. Peers also join and leave, sit behind NATs, have unequal bandwidth, and may be untrusted. Discovery, integrity, incentives, and availability therefore become core protocol concerns rather than optional deployment details.

![Client-server communication concentrates service at a server, whereas peers can exchange directly.](assets/client-server-vs-p2p.png){fig-alt="Client server model with two clients contacting one server and peer-to-peer model with two equal peers exchanging data" width="64%"}

*Figure source: [Michel Bakni, Client-server vs peer-to-peer, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Client-server_Vs_peer-to-peer_-_en.png), licensed under CC BY-SA 4.0.*

A hybrid system is common. A central service authenticates users and introduces peers, while bulk data moves directly. BitTorrent can use a tracker or DHT for discovery and peers for pieces. WebRTC uses an application signaling service to exchange session metadata and then attempts a direct media path.

#### **Processes, Sockets, and Transport Selection**

An application protocol runs between processes through sockets. Its transport choice follows semantics:

| Requirement | Consequence |
|---|---|
| reliable ordered byte stream | TCP is a natural substrate, but the application must frame messages |
| independent reliable streams and migration | QUIC can avoid cross-stream transport head-of-line blocking |
| self-contained datagrams and timing control | UDP exposes message boundaries but the application must add every needed guarantee |
| browser deployment | HTTP APIs, WebSocket, WebTransport, or WebRTC may be available while raw sockets are not |

Using UDP does not remove congestion-control responsibility, and using TCP does not make an operation safe to retry. Transport properties and application semantics solve different problems.

#### **Stateful and Stateless Protocols**

A **stateless interaction** carries enough information for the server to interpret each request independently. This improves retry, failover, caching, and horizontal scaling. It does not mean that the application has no state: a shopping cart may live in a database or signed client token rather than in one process's memory.

A **stateful protocol** retains conversation context such as authentication state, selected mailbox, subscription, transaction, or media session. State can reduce repeated metadata and support long-lived interaction, but failover must preserve or reconstruct it. A good design states the scope and lifetime of each state variable instead of calling the complete system simply "stateless."

#### **Distribution Capacity**

For a file of $F$ bits sent to $N$ clients, server upload $u_s$, slowest client download $d_{min}$, and peer uploads $u_i$, ideal lower bounds are

$$
D_{CS}\ge \max\left(\frac{NF}{u_s},\frac{F}{d_{min}}\right)
$$

and

$$
D_{P2P}\ge \max\left(\frac{F}{u_s},\frac{F}{d_{min}},\frac{NF}{u_s+\sum_i u_i}\right).
$$

The client-server origin must upload $N$ complete copies. In P2P, the origin must inject at least one copy and the group must collectively upload $NF$ bits. These are lower bounds, not promises: piece availability, churn, protocol overhead, asymmetric access links, incentives, and topology can dominate an implementation.

In [1]:
def distribution_lower_bounds(file_gib, clients, server_upload_mbps,
                              slowest_download_mbps, peer_upload_mbps):
    """Return ideal client-server and P2P lower bounds in minutes."""
    file_bits = file_gib * (1024 ** 3) * 8
    us = server_upload_mbps * 1_000_000
    dmin = slowest_download_mbps * 1_000_000
    peer_sum = clients * peer_upload_mbps * 1_000_000

    client_server = max(clients * file_bits / us, file_bits / dmin)
    peer_to_peer = max(
        file_bits / us,
        file_bits / dmin,
        clients * file_bits / (us + peer_sum),
    )
    return client_server / 60, peer_to_peer / 60


print("clients | client-server lower bound | P2P lower bound")
for clients in (10, 100, 1000):
    cs, p2p = distribution_lower_bounds(
        file_gib=1,
        clients=clients,
        server_upload_mbps=100,
        slowest_download_mbps=50,
        peer_upload_mbps=10,
    )
    print(f"{clients:7d} | {cs:23.1f} min | {p2p:14.1f} min")

clients | client-server lower bound | P2P lower bound
     10 |                    14.3 min |            7.2 min
    100 |                   143.2 min |           13.0 min
   1000 |                  1431.7 min |           14.2 min


The client-server bound grows linearly once origin upload is the bottleneck. The P2P bound approaches a value determined by aggregate peer contribution. A CDN changes the client-server calculation by adding managed edge upload and cached replicas rather than by turning users into peers.

### **Designing an Application Protocol**

#### **Message Syntax and Semantics**

A protocol needs both **syntax** and **semantics**. Syntax says how to recognize fields; semantics says what the fields mean and which state transition follows. A valid JSON object is not yet a protocol. The peers still need a version, operation names, identifiers, ordering rules, error model, limits, and retry behavior.

A useful request envelope often contains:

- a protocol version and message type;
- a request or correlation identifier;
- operation-specific payload;
- optional deadline, authorization context, and idempotency key;
- an explicit error representation that distinguishes invalid input, conflict, unavailable service, and internal failure.

Unknown fields and versions need a policy. Silently interpreting a new field with old semantics can be worse than rejecting it. [RFC 9205](https://www.rfc-editor.org/rfc/rfc9205) gives broader guidance for building application protocols using HTTP, including method choice, status codes, caching, redirects, and version evolution.

#### **Framing Messages over Byte Streams**

TCP can return half a message or several messages in one `recv()`. A byte-stream protocol therefore needs framing. Common strategies are:

- fixed-size records;
- delimiter termination, with escaping or a grammar that excludes the delimiter;
- a fixed-width length prefix followed by exactly that many bytes;
- a self-describing encoding whose parser can determine one complete value;
- connection close as an end marker, suitable only when no later message is needed.

![A length-prefixed protocol reconstructs messages from arbitrary TCP receive chunks.](assets/application-protocol-framing.svg){fig-alt="Sender serializes length type request identifier and payload while receiver buffers arbitrary TCP chunks, validates the length and dispatches complete frames" width="98%"}

A peer controls declared lengths, nesting, and field counts, so parsers must set limits before allocating memory. Length is usually encoded in **network byte order**. The decoder below accepts arbitrary chunks and can return zero, one, or many complete frames per call.

In [2]:
import json
import struct


HEADER = struct.Struct("!IBQ")  # body length, one-byte type, eight-byte request ID
MAX_BODY = 1_000_000


def encode_frame(message_type, request_id, payload):
    payload_bytes = json.dumps(payload, separators=(",", ":")).encode("utf-8")
    body_length = 1 + 8 + len(payload_bytes)
    return HEADER.pack(body_length, message_type, request_id) + payload_bytes


class FrameDecoder:
    def __init__(self, max_body=MAX_BODY):
        self.buffer = bytearray()
        self.max_body = max_body

    def feed(self, chunk):
        self.buffer.extend(chunk)
        messages = []

        while len(self.buffer) >= HEADER.size:
            body_length, message_type, request_id = HEADER.unpack_from(self.buffer)
            if body_length < 9 or body_length > self.max_body:
                raise ValueError(f"invalid body length: {body_length}")

            frame_length = 4 + body_length
            if len(self.buffer) < frame_length:
                break  # a partial frame remains buffered

            payload_bytes = bytes(self.buffer[HEADER.size:frame_length])
            del self.buffer[:frame_length]
            messages.append(
                (message_type, request_id, json.loads(payload_bytes.decode("utf-8")))
            )

        return messages


wire = (
    encode_frame(1, 42, {"op": "put", "value": 7})
    + encode_frame(2, 43, {"op": "get"})
)
decoder = FrameDecoder()
chunks = [wire[:3], wire[3:14], wire[14:29], wire[29:]]

for index, chunk in enumerate(chunks, start=1):
    print(f"chunk {index}: {len(chunk):2d} bytes -> {decoder.feed(chunk)}")
print("bytes still buffered:", len(decoder.buffer))

chunk 1:  3 bytes -> []
chunk 2: 11 bytes -> []
chunk 3: 15 bytes -> []
chunk 4: 31 bytes -> [(1, 42, {'op': 'put', 'value': 7}), (2, 43, {'op': 'get'})]
bytes still buffered: 0


#### **Serialization and Content Types**

Serialization converts application values into bytes. Text formats such as JSON are inspectable and widely supported but need conventions for numbers, timestamps, binary data, and unknown fields. Binary formats such as CBOR, MessagePack, or Protocol Buffers can be smaller and schema-aware, but require tooling and compatible schema evolution.

A **content type** identifies how to interpret a representation; it is not decorative metadata. `application/json`, a vendor-specific media type, or a Protobuf schema version tells the receiver which parser and semantics apply. Character encodings must also be explicit. A parser producing values successfully does not prove those values are authorized, within business limits, or meaningful.

#### **Timeouts, Retries, Idempotency, and Backpressure**

A timeout means the caller stopped waiting. It does **not** prove that the server did not execute the operation. If a payment request reached the service but its response was lost, blind retry can charge twice.

An operation is **idempotent** when repeating the same intended operation has the same effect as performing it once. HTTP `PUT` and `DELETE` are defined with idempotent semantics even though repeated responses can differ. A non-idempotent operation such as "increment balance" can be made retry-safe by attaching a stable idempotency key and remembering its completed result.

Retries need a bounded count, backoff, jitter, and deadline. Otherwise a struggling dependency creates more load and a **retry storm**. **Backpressure** makes overload visible upstream: a bounded queue can reject, delay, or shed work instead of consuming unbounded memory.

```text
CALL operation(request_id, deadline)
    repeat while deadline remains and attempts are bounded
        send the same request_id
        if a final response arrives: return it
        wait with exponential backoff plus random jitter
    report unknown outcome when execution cannot be ruled out

SERVER(request_id, operation)
    if request_id already completed: return stored result
    if admission queue is full: return overload signal
    execute once, atomically store result by request_id, return result
```

#### **Text, Binary, and Schema-Based Protocols**

| Design | Strength | Cost | Suitable use |
|---|---|---|---|
| line-oriented text | easy manual debugging | escaping and ambiguous grammar can grow complex | commands and simple administration |
| JSON over framed transport | broad ecosystem and readable payload | larger payload and loose numeric/schema conventions | public APIs and heterogeneous clients |
| binary TLV | unknown fields can be skipped | custom parser and specification burden | compact extensible systems protocols |
| schema-generated binary | compact, typed, tooling support | schema lifecycle and generated-code dependency | internal RPC and high-volume services |

The choice is less important than bounded parsing, precise semantics, compatibility rules, and observability.

In [3]:
from collections import deque


class IdempotentCounterService:
    def __init__(self, queue_capacity=2):
        self.value = 0
        self.completed = {}
        self.pending = deque(maxlen=queue_capacity)

    def submit(self, request_id, amount):
        if request_id in self.completed:
            return "replayed", self.completed[request_id]
        if len(self.pending) == self.pending.maxlen:
            return "backpressure", None
        self.pending.append((request_id, amount))
        return "accepted", None

    def process_one(self):
        request_id, amount = self.pending.popleft()
        self.value += amount
        self.completed[request_id] = self.value
        return request_id, self.value


service = IdempotentCounterService(queue_capacity=2)
print(service.submit("payment-42", 10))
print(service.submit("other", 5))
print(service.submit("too-many", 1))
print("processed:", service.process_one())

# The caller timed out and repeats the same logical operation.
print("retry before completion record is queried:", service.submit("payment-42", 10))
print("processed:", service.process_one())
print("retry after completion:", service.submit("payment-42", 10))
print("final counter:", service.value)

('accepted', None)
('accepted', None)
('backpressure', None)
processed: ('payment-42', 10)
retry before completion record is queried: ('replayed', 10)
processed: ('other', 15)
retry after completion: ('replayed', 10)
final counter: 15


The stable request ID prevents the completed operation from being applied twice. The bounded queue rejects excess work instead of silently growing. A production service must store the result atomically with the effect; an in-memory dictionary is not durable across process failure.

### **Domain Name System**

#### **Names, Addresses, and the DNS Namespace**

An IP address identifies a network endpoint at one moment. A domain name provides a delegated, human-manageable identifier that can outlive one address and map to several kinds of data. DNS is a distributed database indexed by a hierarchical name, record type, and class. It is not merely an Internet-wide dictionary from names to IPv4 addresses.

The fully qualified domain name `www.example.com.` is read from right to left for delegation: the root (`.`), top-level domain `com`, delegated zone `example.com`, and owner name `www`. A **domain** is a subtree of the namespace. A **zone** is the portion served authoritatively under one administrative delegation; a zone can delegate children and therefore need not contain its complete conceptual subtree.

![The DNS namespace is a tree containing delegation boundaries, zones, names, and host labels.](assets/dns-tree.png){fig-alt="DNS tree with root, top-level domains, delegated zones and host labels including a highlighted path to a web host" width="90%"}

*Figure source: [Sylvain Leroux, DNS Tree, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:DNS_Tree.svg), licensed under CC BY-SA 3.0 or GFDL.*

[RFC 1034](https://www.rfc-editor.org/rfc/rfc1034) describes DNS concepts and facilities; [RFC 1035](https://www.rfc-editor.org/rfc/rfc1035) defines core message and record formats.

#### **Root, TLD, and Authoritative Name Servers**

Root servers know delegations for top-level zones. A `.com` server knows delegations beneath `.com`; it normally does not know the final `A` record for every web host. An **authoritative server** answers from zone data for which it has authority. A **recursive resolver** performs work on behalf of clients and caches results.

These are roles, not necessarily single machines. Root service letters, public recursive services, and authoritative providers use replicas, load balancing, and often anycast. Redundancy also requires operational independence: two NS records pointing to names on one failing network are not two effective failure domains.

#### **Recursive and Iterative Resolution**

A laptop's stub resolver usually sends one **recursive** query: return a final answer or an error. On a cache miss, the recursive resolver performs **iterative** queries. Each referral points it closer to the authoritative zone.

![A recursive resolver follows root, TLD, and authoritative referrals, then caches the final RRset by TTL.](assets/dns-resolution-sequence.svg){fig-alt="Browser sends a recursive DNS query to a caching resolver which asks root, dot com, and example dot com authoritative servers iteratively before caching and returning the answer" width="98%"}

The resolver may query several authoritative addresses, retry over another transport, follow aliases, validate DNSSEC, and minimize query names. The clean sequence is a reasoning model, not a claim that every lookup always emits exactly three upstream packets.

#### **Resource Records**

A resource record has an owner name, type, class, TTL, and type-specific data. Records are commonly cached and returned as an **RRset** containing all records of one owner, class, and type.

| Type | Meaning | Example use |
|---|---|---|
| `A` | IPv4 address | web service endpoint |
| `AAAA` | IPv6 address | IPv6 service endpoint |
| `NS` | authoritative name server for a zone | delegation |
| `CNAME` | owner is an alias for another canonical name | redirect lookup within DNS semantics |
| `MX` | mail exchanger with preference | route SMTP delivery for a domain |
| `SOA` | zone authority and timing metadata | serial, primary server, negative-cache basis |
| `TXT` | arbitrary text strings | domain policy and verification data |
| `SRV` | service target, port, priority, and weight | discover named services |
| `CAA` | certification-authority authorization policy | constrain certificate issuance policy |

A CNAME does not issue an HTTP redirect and an MX record does not contain a mailbox. Record types have precise consumers and semantics.

#### **Caching, TTLs, Negative Answers, and Consistency**

The TTL is a maximum cache lifetime supplied with an RRset. Before expiry, a resolver may reuse the cached data without contacting authority. After expiry, it may query again; expiry does not force every cache worldwide to refresh at the same instant.

DNS therefore provides bounded staleness rather than instantaneous global consistency. Lowering a TTL **before** a planned migration can shorten later cache lifetimes, but already cached records retain the TTL they received. Very short TTLs increase resolver and authoritative load and do not eliminate application, browser, or connection caching.

Negative information is cached too. `NXDOMAIN` means the queried name does not exist; `NOERROR` with no requested data means the name may exist but not with that type. [RFC 2308](https://www.rfc-editor.org/rfc/rfc2308) defines negative DNS caching, while [RFC 9520](https://www.rfc-editor.org/rfc/rfc9520) strengthens caching requirements for resolution failures such as repeated `SERVFAIL` or timeout conditions.

#### **Load Distribution and Anycast DNS**

DNS can return several addresses, weighted service records, or answers selected by resolver location and policy. This is coarse request steering. The application may reuse an address long after the DNS answer, and the recursive resolver's location may differ from the client.

With **anycast**, several server instances advertise the same IP prefix and routing directs a client toward one reachable instance. Anycast is valuable for recursive, authoritative, and CDN edge services, but "nearest" means the routing system's selected path, not guaranteed geographic distance or lowest application latency. Stateful service must also tolerate route changes that move later packets to another site.

In [4]:
class DnsCache:
    """Small deterministic TTL cache for positive and negative DNS data."""
    def __init__(self):
        self.entries = {}

    def put(self, name, record_type, value, ttl, now):
        key = (name.rstrip(".").lower(), record_type.upper())
        self.entries[key] = (value, now + ttl)

    def get(self, name, record_type, now):
        key = (name.rstrip(".").lower(), record_type.upper())
        entry = self.entries.get(key)
        if entry is None:
            return "MISS", None, 0
        value, expires_at = entry
        if now >= expires_at:
            del self.entries[key]
            return "EXPIRED", None, 0
        return "HIT", value, expires_at - now


cache = DnsCache()
cache.put("www.example.com.", "A", ["203.0.113.20"], ttl=60, now=0)
cache.put("missing.example.com.", "A", "NXDOMAIN", ttl=30, now=0)

for now in (0, 20, 35, 70):
    positive = cache.get("WWW.EXAMPLE.COM", "a", now)
    negative = cache.get("missing.example.com", "A", now)
    print(f"t={now:2d}s positive={positive}; negative={negative}")

t= 0s positive=('HIT', ['203.0.113.20'], 60); negative=('HIT', 'NXDOMAIN', 30)
t=20s positive=('HIT', ['203.0.113.20'], 40); negative=('HIT', 'NXDOMAIN', 10)
t=35s positive=('HIT', ['203.0.113.20'], 25); negative=('EXPIRED', None, 0)
t=70s positive=('EXPIRED', None, 0); negative=('MISS', None, 0)


At 35 seconds the negative result has expired while the positive RRset remains reusable. At 70 seconds both require a new resolution attempt. The cache key includes name and type; an `A` answer cannot satisfy an `AAAA` query.

### **The Web and HTTP Semantics**

#### **URLs, Requests, Responses, Methods, and Status Codes**

A URL identifies how to locate a resource. In

```text
https://shop.example:8443/products/42?view=compact#reviews
```

`https` is the scheme, `shop.example` the host, `8443` an explicit port, `/products/42` the path, `view=compact` the query, and `reviews` the fragment. The fragment is interpreted by the user agent and is not sent in the HTTP request target.

HTTP is a request-response protocol whose semantics are standardized in [RFC 9110](https://www.rfc-editor.org/rfc/rfc9110). A method communicates intent:

| Method | Safe | Idempotent | Typical meaning |
|---|---:|---:|---|
| `GET` | yes | yes | retrieve a representation |
| `HEAD` | yes | yes | retrieve response metadata without the selected body |
| `POST` | no | no by default | process submitted data under resource-specific semantics |
| `PUT` | no | yes | create or replace the state of the target resource |
| `DELETE` | no | yes | request removal of the target resource |
| `PATCH` | no | not necessarily | apply a partial modification defined by its media type |

Safe means the client did not request a state change, not that the server performs no logging. Idempotent describes intended effect, not identical status codes. Retrying a `DELETE` may first return `204` and later `404` while still satisfying idempotency.

Status classes summarize outcomes: `1xx` informational, `2xx` successful, `3xx` redirection, `4xx` client-side request conditions, and `5xx` server inability. Applications should interpret specific codes and response metadata rather than replacing every error with `200` and an informal JSON flag.

#### **Headers, Representation Metadata, and Content Negotiation**

Headers carry message metadata: `Host` selects an origin, `Content-Type` describes a body, `Content-Length` frames it when applicable, `Accept` expresses response media preferences, and validators support caching. Hop-by-hop fields apply to one transport connection; end-to-end representation metadata must survive intermediaries according to HTTP rules.

Content negotiation can select language, encoding, or media type. If `Accept-Language` changes the response, a shared cache needs `Vary: Accept-Language` or a URL design that makes variants distinct. Otherwise one user's representation may be incorrectly reused for another request.

#### **Cookies, Sessions, and Authentication State**

HTTP requests are independent, but applications often need a session. A response can send `Set-Cookie`; the user agent later selects matching cookies by domain, path, expiry, and policy and adds a `Cookie` request header. The cookie usually carries a session identifier or compact state, not the server's entire in-memory session.

Cookies are ambient state and must be scoped deliberately. `Secure`, `HttpOnly`, and `SameSite` influence transport and browser exposure; they do not by themselves authenticate a user or authorize an operation. Chapter 8 covers attacks and defenses in detail.

#### **Persistent Connections and Pipelining**

HTTP/1.1 normally reuses a TCP connection for several requests, avoiding a handshake and slow-start restart for every object. **Pipelining** permits sending later requests before earlier responses complete, but responses on that connection must remain ordered. A slow first response therefore blocks later ones at the HTTP layer. Browsers historically opened several connections instead, increasing handshake and congestion-state overhead.

The example below uses standard URL and email-header parsers to separate URL components and message fields. It does not connect to the Internet.

In [5]:
from email.parser import Parser
from urllib.parse import urlsplit


def parse_http_text(raw_message, is_response=False):
    """Parse a teaching HTTP/1.x start line and RFC-style header block."""
    normalized = raw_message.replace("\r\n", "\n")
    head, _, body = normalized.partition("\n\n")
    start_line, header_text = head.split("\n", 1)
    headers = Parser().parsestr(header_text)
    return start_line, dict(headers.items()), body


url = urlsplit("https://shop.example:8443/products/42?view=compact#reviews")
print("URL components:")
print({
    "scheme": url.scheme,
    "host": url.hostname,
    "port": url.port,
    "path": url.path,
    "query": url.query,
    "fragment_not_sent": url.fragment,
})

request = (
    "GET /products/42?view=compact HTTP/1.1\r\n"
    "Host: shop.example:8443\r\n"
    "Accept: application/json\r\n"
    "Accept-Language: en-AU\r\n"
    "Cookie: session=abc123\r\n\r\n"
)
response = (
    "HTTP/1.1 200 OK\r\n"
    "Content-Type: application/json\r\n"
    "Cache-Control: private, max-age=60\r\n"
    "Vary: Accept-Language\r\n\r\n"
    '{"id":42,"name":"router"}'
)

print("\nrequest:", parse_http_text(request))
print("response:", parse_http_text(response, is_response=True))

URL components:
{'scheme': 'https', 'host': 'shop.example', 'port': 8443, 'path': '/products/42', 'query': 'view=compact', 'fragment_not_sent': 'reviews'}

request: ('GET /products/42?view=compact HTTP/1.1', {'Host': 'shop.example:8443', 'Accept': 'application/json', 'Accept-Language': 'en-AU', 'Cookie': 'session=abc123'}, '')
response: ('HTTP/1.1 200 OK', {'Content-Type': 'application/json', 'Cache-Control': 'private, max-age=60', 'Vary': 'Accept-Language'}, '{"id":42,"name":"router"}')


The fragment is visible in the parsed URL but absent from the request target. `Vary` tells a cache that language participates in selecting the stored response. A production implementation must also handle binary bodies, transfer codings, informational responses, trailers, limits, malformed input, and HTTP version-specific framing; use a maintained HTTP library rather than this teaching parser.

### **HTTP Evolution**

HTTP/1.1, HTTP/2, and HTTP/3 preserve core methods, status codes, fields, URLs, and caching semantics. They differ in message framing, stream multiplexing, compression state, and transport.

![HTTP versions change multiplexing and the scope of head-of-line blocking.](assets/http-evolution.svg){fig-alt="Three panels compare ordered HTTP 1.1 responses, interleaved HTTP 2 frames over one TCP connection, and independent HTTP 3 streams over QUIC" width="98%"}

#### **HTTP/1.1 and Head-of-Line Blocking**

[RFC 9112](https://www.rfc-editor.org/rfc/rfc9112) defines HTTP/1.1 message syntax and connection management. Messages are textual at the start line and field level. Persistent connections amortize setup, but one connection still carries an ordered response sequence. Multiple parallel TCP connections reduce that application ordering bottleneck at the cost of extra setup and independent congestion-control competition.

#### **HTTP/2 Framing, Streams, and Header Compression**

[RFC 9113](https://www.rfc-editor.org/rfc/rfc9113) defines HTTP/2. It splits messages into binary frames tagged with stream identifiers, permitting request and response frames from many streams to interleave on one TCP connection. This removes HTTP/1.1 response-order head-of-line blocking and reduces the need for many connections.

HPACK compresses repeated fields using static and dynamic tables. Compression state improves efficiency but requires strict size and synchronization controls. HTTP/2 flow control operates per stream and per connection; it is distinct from TCP congestion control and protects receiver/application capacity rather than the network path.

All streams still sit inside one ordered TCP byte stream. If one TCP segment is missing, TCP cannot expose later bytes to the HTTP/2 framing layer even when they belong to another stream. That is **transport head-of-line blocking**.

#### **HTTP/3 over QUIC**

[RFC 9114](https://www.rfc-editor.org/rfc/rfc9114) maps HTTP semantics onto QUIC streams. A missing QUIC packet can delay frames that depend on its missing stream data, while complete data on another stream can remain deliverable. HTTP/3 also uses QPACK, whose design controls how header-compression dependencies interact with independent streams.

HTTP/3 does not make loss free. Streams share a connection's path, congestion controller, and endpoint resources. Loss can lower the aggregate sending rate, and application dependencies can still make one object wait for another.

#### **Connection Reuse, Prioritization, and Web Performance**

Connection reuse avoids repeated DNS, transport, and cryptographic setup only when origin, certificate, protocol, address policy, and server behavior permit it. A warm HTTP/2 or HTTP/3 connection can carry a new request immediately; a cold path may need name resolution and handshake work first.

Multiplexing also needs prioritization. HTML, render-blocking CSS, a hero image, analytics, and an off-screen video are not equally urgent. [RFC 9218](https://www.rfc-editor.org/rfc/rfc9218) defines extensible HTTP priority signals, but scheduling remains a cooperative hint and implementation choice rather than a hard delivery deadline.

The model below isolates blocking scope. Times are synthetic and include one 50 ms connection setup. A loss delays resource A by 80 ms.

In [6]:
def simplified_http_completion(resource_ms, setup_ms=50, loss_resource="A", recovery_ms=80):
    """Compare completion times under intentionally simplified blocking rules."""
    names = list(resource_ms)

    # HTTP/1.1 pipelined responses must complete in request order.
    h1 = {}
    clock = setup_ms
    for name in names:
        clock += resource_ms[name]
        if name == loss_resource:
            clock += recovery_ms
        h1[name] = clock

    # HTTP/2 streams overlap, but one TCP loss delays delivery for all streams.
    h2 = {
        name: setup_ms + resource_ms[name] + recovery_ms
        for name in names
    }

    # HTTP/3 streams overlap; this model charges recovery only to affected A.
    h3 = {
        name: setup_ms + resource_ms[name] + (recovery_ms if name == loss_resource else 0)
        for name in names
    }
    return h1, h2, h3


resources = {"A": 120, "B": 40, "C": 60}
h1, h2, h3 = simplified_http_completion(resources)
print("resource | HTTP/1.1 pipeline | HTTP/2 + TCP loss | HTTP/3 stream loss")
for name in resources:
    print(f"{name:8s} | {h1[name]:17d} ms | {h2[name]:17d} ms | {h3[name]:17d} ms")

resource | HTTP/1.1 pipeline | HTTP/2 + TCP loss | HTTP/3 stream loss
A        |               250 ms |               250 ms |               250 ms
B        |               290 ms |               170 ms |                90 ms
C        |               350 ms |               190 ms |               110 ms


This is not a protocol benchmark. Real completion depends on packetization, congestion windows, frame scheduling, dependencies, server computation, connection reuse, and which bytes were lost. The table exists to make the blocking domains explicit.

### **Caching, Proxies, and Content Delivery Networks**

#### **Browser and Shared Caches**

A cache stores a response so a later request can reuse it without transferring the same representation from the origin. A browser cache is **private** to one user agent. A proxy or CDN cache is **shared** among requests and must obey stricter rules for personalized or authorized responses.

The cache key includes at least method and target URI, often plus request fields named by `Vary`, and implementations may partition by top-level site for privacy. A cache hit is valid only if the stored response is suitable for the new request and may be reused under [RFC 9111](https://www.rfc-editor.org/rfc/rfc9111).

#### **Validation, Freshness, and Cache-Control**

**Freshness** permits reuse without contacting the origin. `Cache-Control: max-age=300` gives a five-minute freshness lifetime relative to response time and age calculations. Important directives include:

| Directive | Meaning |
|---|---|
| `no-store` | do not store this response in a cache |
| `no-cache` | storage is allowed, but revalidate before reuse |
| `private` | shared caches must not store it; a private cache may |
| `public` | explicitly permits shared caching where other rules allow |
| `max-age` | freshness lifetime for caches |
| `s-maxage` | shared-cache freshness overriding `max-age` |
| `must-revalidate` | do not reuse stale content without successful validation when required |
| `immutable` | fresh representation is not expected to change |

An `ETag` is a representation validator. A stale cache can send `If-None-Match`; `304 Not Modified` refreshes metadata while reusing the stored body. `Last-Modified` and `If-Modified-Since` provide a time-based validator with lower precision. `no-cache` therefore does not mean "never save"; `no-store` does.

#### **Forward and Reverse Proxies**

A **forward proxy** acts for clients: it may enforce policy, filter, cache, or provide an egress identity. A **reverse proxy** acts in front of origins: it terminates application connections, routes requests, balances replicas, caches, compresses, and shields internal topology. In both cases, HTTP fields and client identity must be handled deliberately; blindly trusting forwarding headers creates ambiguity and security risk.

#### **CDN Request Mapping and Edge Caches**

A Content Delivery Network places cache and serving capacity near many access networks. Request mapping can use DNS answers, anycast routing, application redirects, and internal load balancing. The chosen edge balances geography, network path, current health, cache contents, and cost; it is not necessarily the physically nearest building.

![A CDN replaces one distant serving point with a distributed set of managed edge locations.](assets/cdn-comparison.png){fig-alt="Side-by-side comparison of one central content server serving all clients and a distributed CDN with several edge servers serving nearby clients" width="94%"}

*Figure source: [D. Ilyin after Kanoha, Single server versus CDN, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:NCDN_-_CDN.svg), released under CC0.*

#### **Origin Shielding, Replication, and Cache Invalidation**

An **origin shield** is an intermediate cache selected to consolidate misses from many edges. If one popular object expires at thousands of edges, request coalescing and a shield can turn a thundering herd into one origin validation rather than thousands of simultaneous fetches.

![Fresh hits, conditional validation, and misses traverse different portions of a CDN path.](assets/cache-cdn-validation.svg){fig-alt="Browser, CDN edge, origin shield and origin show a fresh cache hit, ETag revalidation with 304, and a miss returning 200 and filling caches" width="98%"}

Invalidation is a distributed consistency problem. Options include short freshness, explicit purge, versioned content URLs, and revalidation. Versioned immutable URLs are ideal for static assets because new content receives a new key. Purging mutable URLs can be fast but not instantaneous across every edge and failure state.

For hit ratio $h$, hit latency $T_h$, and miss latency $T_m$, a first-order average is

$$
E[T]=hT_h+(1-h)T_m.

$$

The average hides tail latency and object popularity. A 95% request hit ratio can coexist with poor byte hit ratio if the misses are giant videos, and a cache can be fast while serving stale or incorrectly keyed content.

In [7]:
from dataclasses import dataclass


@dataclass
class CachedResponse:
    stored_at: int
    max_age: int
    etag: str
    private: bool = False
    no_store: bool = False


def cache_decision(entry, now, shared_cache, request_allows_stale=False):
    """Classify a simplified HTTP cache state."""
    if entry is None or entry.no_store:
        return "MISS"
    if shared_cache and entry.private:
        return "MISS: private response"

    current_age = now - entry.stored_at
    if current_age <= entry.max_age:
        return f"FRESH HIT (Age={current_age}s)"
    if entry.etag:
        return f"REVALIDATE with If-None-Match: {entry.etag}"
    return "STALE: fetch a new representation"


public_asset = CachedResponse(stored_at=100, max_age=60, etag='"v7"')
private_page = CachedResponse(stored_at=100, max_age=60, etag='"user-3"', private=True)

for now in (120, 170):
    print(f"t={now}: edge asset -> {cache_decision(public_asset, now, True)}")
    print(f"t={now}: edge private -> {cache_decision(private_page, now, True)}")
    print(f"t={now}: browser private -> {cache_decision(private_page, now, False)}")

t=120: edge asset -> FRESH HIT (Age=20s)
t=120: edge private -> MISS: private response
t=120: browser private -> FRESH HIT (Age=20s)
t=170: edge asset -> REVALIDATE with If-None-Match: "v7"
t=170: edge private -> MISS: private response
t=170: browser private -> REVALIDATE with If-None-Match: "user-3"


In [8]:
def cache_effect(hit_ratio, edge_latency_ms, miss_latency_ms,
                 requests_per_second, coalesced_misses=1):
    """Compute first-order latency and origin request load."""
    expected_latency = (
        hit_ratio * edge_latency_ms
        + (1 - hit_ratio) * miss_latency_ms
    )
    raw_misses = requests_per_second * (1 - hit_ratio)
    origin_requests = raw_misses / coalesced_misses
    return expected_latency, raw_misses, origin_requests


for hit_ratio in (0.50, 0.90, 0.99):
    latency, misses, origin = cache_effect(
        hit_ratio=hit_ratio,
        edge_latency_ms=18,
        miss_latency_ms=220,
        requests_per_second=10_000,
        coalesced_misses=20,
    )
    print(
        f"hit={hit_ratio:4.0%}: E[latency]={latency:6.1f} ms, "
        f"edge misses={misses:6.0f}/s, origin after collapse={origin:5.1f}/s"
    )

hit= 50%: E[latency]= 119.0 ms, edge misses=  5000/s, origin after collapse=250.0/s
hit= 90%: E[latency]=  38.2 ms, edge misses=  1000/s, origin after collapse= 50.0/s
hit= 99%: E[latency]=  20.0 ms, edge misses=   100/s, origin after collapse=  5.0/s


The model shows why the final percentage points of hit ratio matter at high request volume and why miss coalescing protects an origin. It does not price stale content, purge delay, large-object byte volume, or regional failure.

### **Electronic Mail and Messaging**

#### **SMTP, Message Transfer, and Relay**

Email is a store-and-forward application rather than one end-to-end socket from the sender's laptop to the recipient's laptop. A Mail User Agent (MUA) submits a message to a Mail Submission Agent (MSA). Mail Transfer Agents (MTAs) look up the recipient domain's MX records and relay the message with SMTP. A delivery agent places it in the recipient's mailbox.

![Email submission, SMTP relay, local delivery, and retrieval are separate application roles.](assets/email-delivery.png){fig-alt="Source user agent submits mail to a source provider, SMTP transfers it to a destination provider, and a destination client retrieves it" width="88%"}

*Figure source: [144p, Schema of e-mail delivery, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Schema_of_e-mail_delivery.svg), licensed under CC BY-SA 4.0.*

[RFC 5321](https://www.rfc-editor.org/rfc/rfc5321) defines SMTP transfer. The SMTP **envelope** contains routing recipients used during delivery; [RFC 5322](https://www.rfc-editor.org/rfc/rfc5322) message headers such as `From`, `To`, and `Subject` are part of message content. They can differ, just as a postal envelope and letter heading can differ.

SMTP responses distinguish success, temporary failure, and permanent failure. A sending MTA queues a temporary failure and retries later with policy and expiry. Accepting a message transfers responsibility; it does not prove that a human read it.

#### **IMAP, POP3, and Mail Access**

SMTP moves outgoing mail between systems. Mail access uses different protocols:

- **IMAP4rev2**, standardized in [RFC 9051](https://www.rfc-editor.org/rfc/rfc9051), keeps server-side mailbox structure and supports synchronized views, flags, searching, and partial retrieval across devices.
- **POP3**, defined by [RFC 1939](https://www.rfc-editor.org/rfc/rfc1939), is a simpler retrieval protocol commonly associated with download-oriented workflows.
- webmail exposes mailbox behavior through HTTP while its provider still participates in SMTP with other domains.

Calling all three "email protocol" hides their different directions and state models.

#### **Message Queues and Delivery Semantics**

Application message brokers generalize store-and-forward delivery. A producer publishes, the broker persists or buffers, and a consumer acknowledges according to a contract.

| Semantics | Typical acknowledgment point | Consequence |
|---|---|---|
| at-most-once | remove before or without retry | no duplicate delivery, but failure can lose work |
| at-least-once | remove only after consumer ACK | work is retried, so duplicates are possible |
| effectively-once effect | at-least-once delivery plus deduplicated/transactional effect | requires stable identity and coordinated state |

"Exactly once" must name the boundary. A broker can avoid delivering one log record twice within its transaction model while an external email, payment, or database effect still repeats after a crash. Idempotent consumers, deduplication keys, transactional outboxes, and offset commits make the effect boundary explicit.

In [9]:
class AtLeastOnceConsumer:
    def __init__(self):
        self.processed_ids = set()
        self.balance = 0

    def handle(self, message):
        """Apply an externally visible effect once by stable message ID."""
        message_id, amount = message
        if message_id in self.processed_ids:
            return "duplicate suppressed"
        self.balance += amount
        self.processed_ids.add(message_id)
        return "effect committed"


consumer = AtLeastOnceConsumer()
deliveries = [
    ("order-17", 25),
    ("order-17", 25),  # ACK was lost, so the broker redelivered
    ("order-18", 10),
]

for message in deliveries:
    print(message, "->", consumer.handle(message))
print("final balance:", consumer.balance)

('order-17', 25) -> effect committed
('order-17', 25) -> duplicate suppressed
('order-18', 10) -> effect committed
final balance: 35


The consumer makes a repeated delivery harmless. Durability still requires the deduplication record and balance effect to commit atomically; two unrelated writes can fail between them.

### **Multimedia and Real-Time Communication**

#### **Stored Streaming and Adaptive Bitrate**

Stored video can tolerate startup buffering and several seconds of delivery variation, but it must avoid long stalls. Encoding one title at several bitrates and resolutions lets a player trade quality for reliability as network conditions change.

Each representation is divided into independently retrievable segments aligned to the same media timeline. Segmenting turns adaptation into repeated object selection over ordinary HTTP and CDN caches rather than requiring the server to maintain one custom continuous stream per viewer.

#### **DASH and Client-Side Adaptation**

In MPEG-DASH, a Media Presentation Description (MPD) lists periods, adaptation sets, representations, codecs, timing, and segment locations. The client chooses; the manifest advertises alternatives but does not know the client's current buffer, viewport, decoder, data budget, or throughput.

![An adaptive player combines a manifest, recent throughput, and playback-buffer state to choose each next segment.](assets/adaptive-streaming-loop.svg){fig-alt="Several aligned video bitrate representations feed a player loop that measures throughput and buffer, chooses a safe bitrate and downloads the next HTTP segment" width="98%"}

For segment duration $L$ seconds, selected bitrate $r_k$, measured download throughput $x_k$, and buffer before download $B_k$, download time is approximately

$$
d_k=\frac{r_kL}{x_k},

$$

and the post-download buffer is

$$
B_{k+1}=\max(0,B_k-d_k)+L.

$$

If $d_k>B_k$, playback stalls for $d_k-B_k$. Choosing the highest bitrate below the last throughput sample is unstable because samples are noisy and a segment can consume the safety margin. Practical ABR algorithms smooth throughput, use buffer thresholds, apply hysteresis, and account for segment size variation.

In [10]:
def choose_bitrate(available_mbps, estimate_mbps, buffer_seconds):
    """A small hybrid throughput/buffer ABR policy."""
    if buffer_seconds < 4:
        return available_mbps[0]
    safety_budget = 0.75 * estimate_mbps
    eligible = [rate for rate in available_mbps if rate <= safety_budget]
    return max(eligible, default=available_mbps[0])


def simulate_abr(throughputs, segment_seconds=4):
    rates = [0.5, 1.5, 3.0, 6.0]
    estimate = throughputs[0]
    buffer_seconds = 4.0
    trace = []

    for index, throughput in enumerate(throughputs, start=1):
        selected = choose_bitrate(rates, estimate, buffer_seconds)
        download_time = selected * segment_seconds / throughput
        stall = max(0.0, download_time - buffer_seconds)
        buffer_seconds = max(0.0, buffer_seconds - download_time) + segment_seconds
        estimate = 0.7 * estimate + 0.3 * throughput
        trace.append((index, throughput, selected, download_time, stall, buffer_seconds, estimate))

    return trace


network = [3.5, 4.0, 2.2, 0.8, 1.1, 4.5, 6.5]
print("seg | network | selected | download | stall | buffer | next estimate")
for row in simulate_abr(network):
    seg, network_rate, selected, download, stall, buffer, estimate = row
    print(
        f"{seg:3d} | {network_rate:6.1f} | {selected:8.1f} |"
        f" {download:8.2f}s | {stall:5.2f}s | {buffer:6.2f}s | {estimate:7.2f}"
    )

seg | network | selected | download | stall | buffer | next estimate
  1 |    3.5 |      1.5 |     1.71s |  0.00s |   6.29s |    3.50
  2 |    4.0 |      1.5 |     1.50s |  0.00s |   8.79s |    3.65
  3 |    2.2 |      1.5 |     2.73s |  0.00s |  10.06s |    3.21
  4 |    0.8 |      1.5 |     7.50s |  0.00s |   6.56s |    2.49
  5 |    1.1 |      1.5 |     5.45s |  0.00s |   5.10s |    2.07
  6 |    4.5 |      1.5 |     1.33s |  0.00s |   7.77s |    2.80
  7 |    6.5 |      1.5 |     0.92s |  0.00s |  10.85s |    3.91


The throughput collapse at segment 4 drains the safety buffer and forces a later downshift. The policy is intentionally small: production players also consider startup phase, codec switching, live-edge latency, abandonment of a slow segment, viewport, device limits, and CDN errors.

#### **RTP, WebRTC, Jitter Buffers, and Interactive Latency**

Interactive audio and video cannot hide several seconds of uncertainty. A late voice packet may be less useful than a missing one. RTP, standardized in [RFC 3550](https://www.rfc-editor.org/rfc/rfc3550), carries sequence numbers, media timestamps, payload types, and source identifiers. RTCP reports reception quality and timing. RTP does not itself guarantee delivery, reserve bandwidth, or define one codec.

A **jitter buffer** delays playout so packets with variable network delay can be reordered before their deadline. A larger buffer hides more jitter but adds mouth-to-ear latency. Missing packets at the deadline use loss concealment rather than blocking indefinitely.

WebRTC combines browser media APIs, negotiated codecs, congestion control, encrypted media, and NAT traversal. Application-specific **signaling** exchanges offers, answers, and ICE candidates; WebRTC deliberately does not standardize that signaling channel.

![WebRTC uses application signaling and ICE candidate checks to select a direct path or TURN relay.](assets/webrtc-ice-paths.svg){fig-alt="Two browsers exchange offer answer and ICE candidates through signaling, use STUN to learn mappings, test a direct media path, and fall back to a TURN relay" width="96%"}

[RFC 8445](https://www.rfc-editor.org/rfc/rfc8445) defines Interactive Connectivity Establishment (ICE). STUN helps an endpoint learn a server-observed mapping; TURN allocates a relay address when direct connectivity fails. A discovered address is only a candidate. ICE connectivity checks validate candidate pairs before selecting a path.

In [11]:
def jitter_buffer_playout(arrival_ms, start_ms, packet_interval_ms=20):
    """Return delivered or concealed packets at fixed playout deadlines."""
    highest_sequence = max(arrival_ms)
    trace = []
    for sequence in range(highest_sequence + 1):
        deadline = start_ms + sequence * packet_interval_ms
        arrival = arrival_ms.get(sequence)
        if arrival is None:
            outcome = "conceal missing packet"
        elif arrival <= deadline:
            outcome = "play packet"
        else:
            outcome = f"late by {arrival - deadline} ms -> conceal"
        trace.append((sequence, arrival, deadline, outcome))
    return trace


# Packet 3 arrives before packet 2; packet 4 never arrives.
arrivals = {0: 30, 1: 45, 2: 105, 3: 75, 5: 118}
for start in (50, 70):
    print(f"\nplayout starts at {start} ms")
    for sequence, arrival, deadline, outcome in jitter_buffer_playout(arrivals, start):
        print(f"seq={sequence}, arrival={str(arrival):>4s}, deadline={deadline:3d} -> {outcome}")


playout starts at 50 ms
seq=0, arrival=  30, deadline= 50 -> play packet
seq=1, arrival=  45, deadline= 70 -> play packet
seq=2, arrival= 105, deadline= 90 -> late by 15 ms -> conceal
seq=3, arrival=  75, deadline=110 -> play packet
seq=4, arrival=None, deadline=130 -> conceal missing packet
seq=5, arrival= 118, deadline=150 -> play packet

playout starts at 70 ms
seq=0, arrival=  30, deadline= 70 -> play packet
seq=1, arrival=  45, deadline= 90 -> play packet
seq=2, arrival= 105, deadline=110 -> play packet
seq=3, arrival=  75, deadline=130 -> play packet
seq=4, arrival=None, deadline=150 -> conceal missing packet
seq=5, arrival= 118, deadline=170 -> play packet


Starting at 70 ms adds latency but saves packet 2; packet 3 was held despite arriving first so media order remains correct. Packet 4 is concealed in either case. An adaptive jitter buffer continuously estimates delay variation rather than choosing one fixed start time forever.

#### **Quality of Experience**

Network metrics become useful only through application outcomes:

| Application | Network evidence | User-facing QoE |
|---|---|---|
| stored video | segment throughput, request errors, CDN RTT | startup time, rebuffer ratio, average quality, switch frequency |
| live stream | segment availability and download delay | distance from live edge, stalls, quality |
| voice/video call | one-way delay, jitter, loss, concealment, bitrate | conversational responsiveness, freezes, intelligibility |
| cloud interaction | input-to-frame delay and loss | control responsiveness and visual stability |

Maximizing bitrate alone can worsen QoE by causing stalls. Minimizing buffer alone can increase loss sensitivity. The objective must reflect the interaction users experience.

### **Peer-to-Peer and Overlay Systems**

#### **BitTorrent and Swarming**

BitTorrent divides a file into verifiable pieces. A **swarm** exchanges different pieces in parallel, so a peer can upload pieces before it owns the complete file. Rarest-first selection improves piece diversity; optimistic unchoking and reciprocal upload policy encourage contribution. Integrity hashes detect corrupt pieces, while metadata authenticity remains a separate trust question.

The original protocol is documented in [BEP 3](https://www.bittorrent.org/beps/bep_0003.html). A tracker can introduce peers, but data transfer remains peer-to-peer. Peers may also discover one another through a DHT.

#### **Distributed Hash Tables**

A **Distributed Hash Table (DHT)** assigns node identifiers and object keys to a logical identifier space. A key is stored or located at one or several responsible nodes. Routing tables keep selected long-distance contacts so lookup can approach the target without knowing every peer.

![A DHT hashes resource names into keys and maps those keys to responsible peers in a distributed table.](assets/dht-concept.png){fig-alt="Hash function maps file names to keys and a distributed hash table maps the keys to peer addresses" width="80%"}

*Figure source: [Jnlin, DHT en, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:DHT_en.svg), released into the public domain.*

In a ring-style design, the successor of key $k$ is the first node encountered clockwise with identifier at least $k$, wrapping at the identifier-space limit. Chord-style finger tables target exponentially increasing offsets, yielding logarithmic lookup state and hop count under ideal stable assumptions. Kademlia, used by BitTorrent's Mainline DHT, organizes distance with XOR and is specified for BitTorrent in [BEP 5](https://www.bittorrent.org/beps/bep_0005.html).

#### **Overlay Routing and NAT Traversal**

An overlay edge is an application relationship carried over the underlying IP network. Two adjacent DHT nodes may be many router hops apart. Overlay optimization must therefore consider physical latency and failure domains, not only logical identifier distance.

NATs and firewalls make unsolicited inbound peer connectivity difficult. Trackers, rendezvous services, hole punching, ICE/STUN, and relays provide introduction or fallback. A relay restores reachability by becoming a traffic intermediary, but consumes provider bandwidth and adds another path segment. Peers also require rate limits, identity validation, and abuse controls because decentralized reachability expands the attack surface discussed in Chapter 8.

In [12]:
from bisect import bisect_left
from hashlib import sha1


RING_SIZE = 128
NODES = [5, 18, 40, 72, 90, 113]


def identifier(text):
    return int.from_bytes(sha1(text.encode()).digest(), "big") % RING_SIZE


def successor(value):
    index = bisect_left(NODES, value)
    return NODES[index] if index < len(NODES) else NODES[0]


def clockwise_distance(start, end):
    return (end - start) % RING_SIZE


def finger_table(node):
    return sorted({successor((node + 2 ** power) % RING_SIZE) for power in range(7)})


def chord_like_route(start, key):
    target = successor(key)
    current = start
    path = [current]

    while current != target:
        remaining = clockwise_distance(current, target)
        candidates = [
            node for node in finger_table(current)
            if 0 < clockwise_distance(current, node) <= remaining
        ]
        current = max(candidates, key=lambda node: clockwise_distance(path[-1], node))
        path.append(current)
    return target, path


for object_name in ("lecture.mp4", "notes.pdf", "dataset.csv"):
    key = identifier(object_name)
    owner, path = chord_like_route(start=5, key=key)
    print(f"{object_name:12s} -> key={key:3d}, owner={owner:3d}, route={path}")

lecture.mp4  -> key= 91, owner=113, route=[5, 72, 113]
notes.pdf    -> key=121, owner=  5, route=[5]
dataset.csv  -> key= 23, owner= 40, route=[5, 40]


The lookup jumps through logical fingers rather than walking every node. Real DHTs replicate data, refresh buckets, verify responses, handle churn, bound amplification, and tolerate malicious peers; the ring model only demonstrates consistent key ownership and multi-hop overlay routing.

### **Building a DNS Client and Minimal HTTP Client**

The next examples expose application wire formats without contacting an external service. Production code should use a maintained resolver and HTTP library: both protocols include transport fallback, compression, security, retries, limits, and edge cases beyond a notebook implementation.

#### **Encoding a DNS Query and Parsing an Answer**

A classic DNS message begins with a 12-byte header. The question contains a label-encoded name, type, and class. Labels are length-prefixed, and a zero byte terminates the name. Responses may compress repeated names with pointers whose top two bits are `11`.

```text
choose unpredictable transaction ID
set RD when recursion is desired
encode each QNAME label with a one-byte length
append QTYPE and QCLASS
send to a configured resolver with timeout and bounded retry
validate source, transaction ID, response bit, question, and RCODE
if TC is set, retry with a suitable reliable DNS transport
parse bounded names and RR lengths; never follow pointer loops
```

The code constructs an `A` query and a synthetic response that uses `c0 0c`, a compression pointer to the question name at byte offset 12.

In [13]:
import ipaddress
import secrets
import socket
import struct


def encode_dns_name(name):
    labels = name.rstrip(".").split(".")
    encoded = bytearray()
    for label in labels:
        part = label.encode("idna")
        if not 0 < len(part) <= 63:
            raise ValueError("DNS label must contain 1..63 encoded bytes")
        encoded.append(len(part))
        encoded.extend(part)
    encoded.append(0)
    return bytes(encoded)


def build_dns_a_query(name, transaction_id=None):
    transaction_id = secrets.randbits(16) if transaction_id is None else transaction_id
    flags = 0x0100  # recursion desired
    header = struct.pack("!HHHHHH", transaction_id, flags, 1, 0, 0, 0)
    question = encode_dns_name(name) + struct.pack("!HH", 1, 1)  # A, IN
    return transaction_id, header + question


def parse_single_a_response(packet, expected_id):
    transaction_id, flags, qd, an, ns, ar = struct.unpack("!HHHHHH", packet[:12])
    if transaction_id != expected_id or not (flags & 0x8000):
        raise ValueError("mismatched transaction or not a response")
    rcode = flags & 0x000F
    if rcode != 0 or qd != 1 or an < 1:
        raise ValueError(f"DNS failure: rcode={rcode}, answers={an}")

    # Skip the uncompressed question name safely for this controlled example.
    offset = 12
    while packet[offset] != 0:
        length = packet[offset]
        offset += 1 + length
    offset += 1 + 4  # terminator plus QTYPE/QCLASS

    name_pointer, rr_type, rr_class, ttl, rdlength = struct.unpack(
        "!HHHIH", packet[offset:offset + 12]
    )
    offset += 12
    if name_pointer != 0xC00C or rr_type != 1 or rr_class != 1 or rdlength != 4:
        raise ValueError("unexpected answer shape")
    return str(ipaddress.IPv4Address(packet[offset:offset + 4])), ttl


txid, query = build_dns_a_query("www.example.com", transaction_id=0x1234)

# Build a deterministic response: copy question, add one compressed A answer.
response_header = struct.pack("!HHHHHH", txid, 0x8180, 1, 1, 0, 0)
question = query[12:]
answer = struct.pack("!HHHIH", 0xC00C, 1, 1, 300, 4) + ipaddress.IPv4Address("203.0.113.20").packed
response = response_header + question + answer

print("query bytes:", query.hex(" "))
print("parsed answer:", parse_single_a_response(response, txid))

query bytes: 12 34 01 00 00 01 00 00 00 00 00 00 03 77 77 77 07 65 78 61 6d 70 6c 65 03 63 6f 6d 00 00 01 00 01
parsed answer: ('203.0.113.20', 300)


This parser intentionally accepts one controlled uncompressed question and one compressed IPv4 answer. A real DNS client must implement bounded pointer traversal, all section counts and RR types, UDP truncation handling, EDNS, internationalized names, DNSSEC policy, multiple resolver addresses, and modern encrypted DNS transports where configured.

#### **Sending One HTTP/1.1 Request to a Local Origin**

The client below starts a one-request loopback server, writes a complete HTTP/1.1 request, reads until connection close, and validates `Content-Length`. It demonstrates the bytes without external network dependency. Closing the connection frames the response for this one-shot example; persistent clients need the full HTTP message-length rules.

In [14]:
import socket
import threading


def local_origin(listener):
    connection, _ = listener.accept()
    with connection:
        request = bytearray()
        while b"\r\n\r\n" not in request:
            chunk = connection.recv(4096)
            if not chunk:
                return
            request.extend(chunk)

        body = b'{"source":"local-origin","ok":true}'
        response = (
            b"HTTP/1.1 200 OK\r\n"
            b"Content-Type: application/json\r\n"
            + f"Content-Length: {len(body)}\r\n".encode()
            + b"Connection: close\r\n\r\n"
            + body
        )
        connection.sendall(response)


with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as listener:
    listener.bind(("127.0.0.1", 0))
    listener.listen(1)
    host, port = listener.getsockname()
    server = threading.Thread(target=local_origin, args=(listener,), daemon=True)
    server.start()

    with socket.create_connection((host, port), timeout=2) as client:
        client.sendall(
            b"GET /status HTTP/1.1\r\n"
            + f"Host: {host}:{port}\r\n".encode()
            + b"Accept: application/json\r\nConnection: close\r\n\r\n"
        )
        wire_response = bytearray()
        while chunk := client.recv(4096):
            wire_response.extend(chunk)

    server.join(timeout=2)

head, body = bytes(wire_response).split(b"\r\n\r\n", 1)
start_line, headers, _ = parse_http_text(head.decode() + "\r\n\r\n", is_response=True)
declared_length = int(headers["Content-Length"])
print(start_line)
print(headers)
print("body length valid:", len(body) == declared_length)
print("body:", json.loads(body))

HTTP/1.1 200 OK
{'Content-Type': 'application/json', 'Content-Length': '35', 'Connection': 'close'}
body length valid: True
body: {'source': 'local-origin', 'ok': True}


### **Tracing a Web Request Through DNS, CDN, and Origin**

A page-load trace is a dependency graph, not a fixed checklist that always pays one cost per layer. Browser caches can bypass DNS and the network. DNS can be warm while the HTTP connection is cold. A reused connection can target an edge whose object is stale. HTML can trigger additional origins, each with its own name, connection, and cache state.

For one resource, a cold path can be reasoned about as:

```text
parse URL and consult service worker/browser cache
resolve origin name through local and recursive DNS caches
select address and establish or reuse an HTTP connection
send request with representation and validation metadata
edge computes cache key and freshness
on miss/stale: edge asks shield; shield may ask origin
origin returns 200 body or 304 validator result
caches update metadata/body according to policy
browser decodes representation and may discover more requests
```

The timing model compares four states. Its numbers are illustrative and sequential; real browsers overlap independent work.

In [15]:
def trace_web_request(browser_hit=False, dns_hit=False, connection_reused=False,
                      edge_hit=False, validator_matches=False):
    events = []
    elapsed = 0.0

    def add(label, duration):
        nonlocal elapsed
        elapsed += duration
        events.append((label, duration, elapsed))

    add("URL parse + browser cache lookup", 0.3)
    if browser_hit:
        add("fresh browser-cache body", 0.2)
        return events

    add("DNS cache lookup" if dns_hit else "recursive DNS resolution", 1.0 if dns_hit else 28.0)
    add("reuse HTTP connection" if connection_reused else "transport + secure setup", 0.2 if connection_reused else 42.0)
    add("request reaches CDN edge", 12.0)

    if edge_hit:
        add("fresh edge-cache response", 7.0)
    elif validator_matches:
        add("shield/origin validation", 38.0)
        add("304 metadata + cached body delivery", 9.0)
    else:
        add("shield/origin miss fetch", 85.0)
        add("body delivery and cache fill", 35.0)

    add("decode representation", 4.0)
    return events


scenarios = {
    "browser hit": dict(browser_hit=True),
    "warm edge": dict(dns_hit=True, connection_reused=True, edge_hit=True),
    "stale validate": dict(dns_hit=True, connection_reused=True, validator_matches=True),
    "cold miss": dict(),
}

for name, settings in scenarios.items():
    trace = trace_web_request(**settings)
    print(f"\n{name}: total={trace[-1][2]:.1f} ms")
    for label, duration, cumulative in trace:
        print(f"  +{duration:5.1f} ms -> {cumulative:6.1f} ms  {label}")


browser hit: total=0.5 ms
  +  0.3 ms ->    0.3 ms  URL parse + browser cache lookup
  +  0.2 ms ->    0.5 ms  fresh browser-cache body

warm edge: total=24.5 ms
  +  0.3 ms ->    0.3 ms  URL parse + browser cache lookup
  +  1.0 ms ->    1.3 ms  DNS cache lookup
  +  0.2 ms ->    1.5 ms  reuse HTTP connection
  + 12.0 ms ->   13.5 ms  request reaches CDN edge
  +  7.0 ms ->   20.5 ms  fresh edge-cache response
  +  4.0 ms ->   24.5 ms  decode representation

stale validate: total=64.5 ms
  +  0.3 ms ->    0.3 ms  URL parse + browser cache lookup
  +  1.0 ms ->    1.3 ms  DNS cache lookup
  +  0.2 ms ->    1.5 ms  reuse HTTP connection
  + 12.0 ms ->   13.5 ms  request reaches CDN edge
  + 38.0 ms ->   51.5 ms  shield/origin validation
  +  9.0 ms ->   60.5 ms  304 metadata + cached body delivery
  +  4.0 ms ->   64.5 ms  decode representation

cold miss: total=206.3 ms
  +  0.3 ms ->    0.3 ms  URL parse + browser cache lookup
  + 28.0 ms ->   28.3 ms  recursive DNS resolution
  + 42

The biggest optimization is often avoiding work, not making every packet slightly faster: a valid browser hit bypasses DNS, connection setup, edge selection, and origin processing. When a network request is necessary, reuse and edge freshness change which dependencies remain on the critical path.

### **Observing and Troubleshooting Application Delivery**

On Windows, useful commands include:

```powershell
Resolve-DnsName example.com -Type A
Resolve-DnsName example.com -Type NS
nslookup -type=mx example.com
curl.exe -vI https://example.com/
curl.exe --http2 -I https://example.com/
Test-NetConnection example.com -Port 443
```

Browser developer tools expose DNS/connect/TLS/request/wait/download timing, protocol version, initiator dependencies, cache status, cookies, and response fields. A `200` can be a cache fill, a `304` successful validation, and a browser memory-cache hit may emit no network packet at all.

Useful Wireshark display filters include:

```text
dns
http
http2
quic
smtp
imap
rtp || rtcp
stun || turnchannel
```

Encrypted application traffic limits payload inspection by design. Endpoint logs, browser timing, resolver logs, CDN headers, server traces, and packet metadata must be correlated. Check the actual cache layer: "HIT" from a browser, edge, or shield describes different work.

| Symptom | Evidence to inspect | Likely classes of cause |
|---|---|---|
| name resolves differently across clients | resolver identity, TTL, answer RRsets, anycast site | cached old data, geo policy, split horizon, resolver failure |
| correct DNS but wrong content | HTTP `Host`/authority, CDN cache key, `Vary`, edge mapping | virtual-host mismatch or cache-key error |
| repeated full downloads | `Cache-Control`, validators, status, request headers | uncacheable response, expired object, missing validator |
| fast network but slow page | request dependency graph and server wait | origin computation, serial application dependencies, cold connections |
| video oscillates or stalls | segment size/time, buffer, selected bitrate, CDN errors | noisy estimator, insufficient safety margin, edge miss |
| call connects only through relay | ICE candidate-pair checks and selected route | NAT/firewall blocks direct path or candidate signaling issue |
| queue consumer repeats effects | message ID, ACK/commit ordering, dedup state | at-least-once redelivery without atomic idempotency |

### **Summary**

- Client-server systems centralize authority; P2P systems turn participants into resources but inherit discovery, churn, NAT, and trust problems.
- An application protocol needs framing, bounded parsing, explicit semantics, retry safety, backpressure, and compatibility rules in addition to a serialization format.
- DNS is a delegated, cached database of typed RRsets. Recursive clients, iterative referrals, TTLs, negative caching, and anycast solve different parts of naming and availability.
- HTTP methods and status codes express application intent. Headers select and describe representations, cookies add scoped state, and persistent connections amortize setup.
- HTTP/2 multiplexes frames over one TCP stream; HTTP/3 maps semantics to QUIC streams and narrows transport head-of-line blocking.
- HTTP caches reuse only semantically valid responses. Validators, `Cache-Control`, `Vary`, proxies, CDN edges, shields, and invalidation jointly determine whether an origin request is necessary.
- SMTP relays messages while IMAP or POP3 accesses mailboxes. General message queues must define acknowledgment, redelivery, and effect boundaries.
- Adaptive streaming trades bitrate against buffer safety. Real-time media trades jitter tolerance against interactive latency and uses ICE/STUN/TURN to establish reachability.
- BitTorrent swarms distribute pieces; DHTs distribute peer or object discovery through an application overlay.

Chapter 8 examines how an adversary changes every assumption made here: names can be forged, sessions stolen, caches poisoned, messages modified, peers impersonated, and metadata observed. Security is therefore not one more application protocol; it constrains the complete path from naming through content consumption.